# P-model and Spatio-Temporal Multifractal Cascade Model (STM-Model 2+1D)

**Autores:** Carlos Eduardo Falanes, Reinaldo Roberto Rosa

**Instituição:** Instituto Nacional de Pesquisas Espaciais - INPE/MCTI

**Contato:** [carlos.falades@inpe.br](mailto:carlos.falades@inpe.br)

---

## Introdução

O **P-model**, proposto por **Meneveau e Sreenivasan (1987)**, é um modelo multifractal em cascata baseado na redistribuição recursiva de energia entre escalas. Esse processo multiplicativo gera estruturas com **auto-similaridade** e **intermitência estatística**, características comuns em sistemas turbulentos.

Neste trabalho, utilizamos a implementação clássica do **P-model 1D** como base para desenvolver uma extensão bidimensional com evolução temporal, denominada **Spatio-Temporal Multifractal Cascade Model (STM-Model 2+1D)**.

---

## P-model 1D

A implementação original do **P-model unidimensional**, desenvolvida por **R.R. Rosa, R. Sautter e N. Joshi**, aplica um processo multiplicativo controlado pelo parâmetro `p`, responsável por introduzir intermitência e comportamento multifractal.

A cada etapa da cascata, a energia é dividida recursivamente em duas partes assimétricas, produzindo uma estrutura hierárquica multifractal.

<div align="center">
  <img src="https://drive.google.com/uc?export=view&id=1bIjLw2ANqh2v9JG6yM4cOJJZGC1lX7XC" width="400"/>
</div>


![Esquema P-model com serie](https://drive.google.com/uc?export=view&id=10qIIuFnMBd40777LPHqhVAwv8BpCdd5V)

Abaixo está o código base utilizado como referência para a generalização:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.mlab as mlab

import matplotlib as mpl

plt.rcParams["figure.dpi"] = 300
mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['font.serif'] = ['Times New Roman']

plt.rcParams.update({
    "font.size": 15,       
    "axes.titlesize": 16,  
    "axes.labelsize": 15,  
    "xtick.labelsize": 14, 
    "ytick.labelsize": 14, 
    "legend.fontsize": 12  
})

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import io

%matplotlib inline

#XP-model adapted from Meneveau & Sreenevasan, 1987 & Malara et al., 2016
#Valid Ranges for p, based on Didier-Sornette's Theory
#Author: R.R.Rosa, R. Sautter and  N. Joshi
#Version: 1.1
#Date: 23/08/2022

import numpy as np
#import statistics as stat
from matplotlib import pyplot


def pmodel(noOrders=5, p=0.5, slope=[]):
    noOrders = int(noOrders)

    dx = np.array([1])
    for n in range(noOrders):
        dx = next_step_1d(dx, p)

    if (slope):
        fourierCoeff = fractal_spectrum_1d(2**noOrders, slope/2)
        meanVal = np.mean(dx)
        stdy = np.std(dx)
        x = np.fft.ifft(dx - meanVal)
        phase = np.angle(x)
        x = fourierCoeff*np.exp(1j*phase)
        x = np.fft.fft(x).real
        x *= stdy/np.std(x)
        x += meanVal
    else:
        x = dx

    return x[0:2**noOrders], dx[0:2**noOrders]


def next_step_1d(dx, p):
    y2 = np.zeros(dx.size*2)
    sign = np.random.rand(1, dx.size) - 0.5
    sign /= np.abs(sign)
    y2[0:2*dx.size:2] = dx + sign*(1-2*p)*dx
    y2[1:2*dx.size+1:2] = dx - sign*(1-2*p)*dx

    return y2


def fractal_spectrum_1d(noValues, slope):
    ori_vector_size = noValues
    ori_half_size = ori_vector_size//2
    a = np.zeros(ori_vector_size)

    for t2 in range(ori_half_size):
        index = t2
        t4 = 1 + ori_vector_size - t2
        if (t4 >= ori_vector_size):
            t4 = t2
        coeff = (index + 1)**slope
        a[t2] = coeff
        a[t4] = coeff

    a[1] = 0

    return a

In [ ]:
#Endogenous (setup: N, p: 0.32-0.42)
#Exogenous (setup: N, p: 0.18-0.28)


N = 10

# Exogenous
p_exo = 0.22
# np.random.seed(126)
np.random.seed(351)
y_exo, dy_exo = pmodel(N, p_exo, 2)

# Endogenous
p_endo = 0.38
# np.random.seed(126)
np.random.seed(351)
y_endo, dy_endo = pmodel(N, p_endo, 2)

def normalize(x):
    x = x + 0.01
    return x / (np.max(x) + 1e-8)

df_exo = normalize(dy_exo)
df_endo = normalize(dy_endo)


fig, axes = plt.subplots(2, 1, figsize=(7, 5), sharex=True)

# Fig. 1a: Endogenous
axes[0].plot(df_endo, lw=0.9)
# axes[0].set_ylim(0, 0.4)
axes[0].set_ylabel("Normalized amplitude")
axes[0].set_title("(a) Endogenous time series (p = {:.2f})".format(p_endo))
axes[0].grid(alpha=0.3)

# Fig. 1b: Exogenous
axes[1].plot(df_exo, lw=0.9)
# axes[1].set_ylim(0, 0.01)
axes[1].set_xlabel("Time steps")
axes[1].set_ylabel("Normalized amplitude")
axes[1].set_title("(b) Exogenous time series (p = {:.2f})".format(p_exo))
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
def prepare_series(
    series,
    peak_percentile=99
):
    normalized_series = (
        (series - np.min(series)) /
        (np.max(series) - np.min(series) + 1e-6)
    )

    df = pd.DataFrame({
        "raw": series,
        "normalized": normalized_series
    })

    threshold = np.percentile(df["normalized"], peak_percentile)
    df["peak"] = (df["normalized"] > threshold).astype(int)
    df["peak_threshold"] = threshold

    return df

In [ ]:
df_endo_peaks = prepare_series(df_endo, peak_percentile=99)
df_exo_peaks  = prepare_series(df_exo,  peak_percentile=99)

fig, axes = plt.subplots(2, 1, figsize=(7, 5), sharex=True)

#  Endogenous
t_endo = np.arange(len(df_endo_peaks))
theta_endo = df_endo_peaks["peak_threshold"].iloc[0]

axes[0].plot(
    t_endo,
    df_endo_peaks["normalized"],
    lw=0.9,
    label="P-model time series"
)

# Threshold line
axes[0].axhline(
    theta_endo,
    linestyle="--",
    linewidth=1.4,
    label="XE threshold (99th percentile)",
    color="orange"
)

# Peaks
axes[0].plot(
    t_endo[df_endo_peaks["peak"] == 1],
    df_endo_peaks.loc[df_endo_peaks["peak"] == 1, "normalized"],
    "x",
    markersize=6,
    label="Extreme events (XE)",
    color="red"
)

axes[0].set_ylabel("Normalized amplitude")
axes[0].set_title("(a) Endogenous time series and extreme event identification (p = {:.2f})".format(p_endo))
axes[0].grid(alpha=0.3)
axes[0].legend(frameon=True)

#  Exogenous
t_exo = np.arange(len(df_exo_peaks))
theta_exo = df_exo_peaks["peak_threshold"].iloc[0]

axes[1].plot(
    t_exo,
    df_exo_peaks["normalized"],
    lw=0.9,
    label="P-model time series"
)

# Threshold line
axes[1].axhline(
    theta_exo,
    linestyle="--",
    linewidth=1.4,
    label="XE threshold (99th percentile)",
    color="orange"
)

# Peaks
axes[1].plot(
    t_exo[df_exo_peaks["peak"] == 1],
    df_exo_peaks.loc[df_exo_peaks["peak"] == 1, "normalized"],
    "x",
    markersize=6,
    label="Extreme events (XE)",
    color="red"
)

axes[1].set_xlabel("Time steps")
axes[1].set_ylabel("Normalized amplitude")
axes[1].set_title("(b) Exogenous time series and extreme event identification (p = {:.2f})".format(p_exo))
axes[1].grid(alpha=0.3)
axes[1].legend(frameon=True)

plt.tight_layout()
plt.show()



## STM-Model (1+1)D

A generalização proposta descreve o modelo como um campo espaço-temporal unidimensional dado por:

$$
A(t,x)
$$

onde `x` representa a dimensão espacial e `t` representa o tempo.

Cada instante `t` corresponde a um campo multifractal unidimensional ao longo de `x`, enquanto a sequência desses campos descreve a evolução temporal do sistema. Dessa forma, o modelo representa uma cascata multifractal **1D no espaço + 1D no tempo**.

Diferentemente do caso puramente bidimensional, aqui a estrutura multifractal evolui dinamicamente, permitindo a incorporação de memória temporal e correlações espaço-temporais.

Essa abordagem é adequada para modelar séries espaciais evolutivas, como sinais turbulentos unidimensionais, dinâmica de interfaces e processos intermitentes ao longo do tempo.

![Esquema STM 1+1D](https://via.placeholder.com/600x300?text=STM+1%2B1D)

---

## Implementação do STM-Model

O modelo foi construído preservando a lógica multiplicativa do **P-model**, agora aplicada ao longo da dimensão espacial, enquanto a evolução temporal é introduzida por um processo multiplicativo dinâmico.

A cada passo de tempo, o campo é atualizado segundo:

$$
A(t+1,x) = W(t,x)\,A(t,x)
$$

onde $W(t,x)$ representa um peso estocástico que pode incorporar dependência espacial local.

A estrutura espacial em cada instante é obtida via uma cascata binária, onde o domínio é subdividido recursivamente e pesos multiplicativos são atribuídos a cada subintervalo.

Dessa forma, o modelo combina:

- uma **cascata multiplicativa 1D no espaço**
- uma **dinâmica multiplicativa no tempo**

resultando em um sistema multifractal espaço-temporal com propriedades emergentes mais próximas de sistemas físicos reais.

In [ ]:
import numpy as np
from scipy.ndimage import gaussian_filter

def stm_model_1d(n, p=0.5, sigma=10):
    field = np.array([[1.0]])

    for _ in range(n):
        field = next_step_1d(field, p, sigma)


    field = gaussian_filter(field, sigma=sigma)

    return field


def next_step_1d(field, p, sigma):
    nx, ny = field.shape
    y2 = np.zeros((2*nx, 2*ny))

    for i in range(nx):
        for j in range(ny):

            val = field[i, j]

            # embaralha orientação (analogia ao sign)
            if np.random.rand() < 0.5:
                weights = 2*np.array([
                    [p*p, p*(1-p)],
                    [(1-p)*p, (1-p)*(1-p)]
                ])
            else:
                weights = 2*np.array([
                    [(1-p)*(1-p), (1-p)*p],
                    [p*(1-p), p*p]
                ])

            # distribui o valor
            y2[2*i:2*i+2, 2*j:2*j+2] = val * weights

    return y2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# --- sua simulação ---
p_exo = 0.18
np.random.seed(351)
df_exo = stm_model_1d(8, p_exo)

p_endo = 0.48
np.random.seed(351)
df_endo = stm_model_1d(8, p_endo)

# normalização
df_exo_norm = (df_exo - df_exo.min()) / (df_exo.max() - df_exo.min())
df_endo_norm = (df_endo - df_endo.min()) / (df_endo.max() - df_endo.min())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from mpl_toolkits.mplot3d import Axes3D

# --- parâmetros ---
split = 0.3
n = 256

# --- colormap consistente ---
n_low = int(n * split)
n_high = n - n_low

colors_low = plt.cm.gray_r(np.linspace(0, 1, n_low))
colors_high = plt.cm.inferno(np.linspace(0, 1, n_high))

colors = np.vstack((colors_low, colors_high))
custom_cmap = mcolors.LinearSegmentedColormap.from_list("custom_map", colors)

norm = mcolors.TwoSlopeNorm(vmin=0, vcenter=split, vmax=1)

# --- grid para 3D ---
nx, ny = df_exo_norm.shape
X, Y = np.meshgrid(np.arange(nx), np.arange(ny))

# --- figura ---
fig = plt.figure(figsize=(12, 10))

# ===== 2D plots =====
ax1 = fig.add_subplot(2, 2, 1)
im1 = ax1.imshow(df_exo_norm, cmap=custom_cmap, norm=norm, origin='lower', aspect='auto')
ax1.set_title(f'(a) Exogenous ($p={p_exo}$)')

ax2 = fig.add_subplot(2, 2, 2)
im2 = ax2.imshow(df_endo_norm, cmap=custom_cmap, norm=norm, origin='lower', aspect='auto')
ax2.set_title(f'(b) Endogenous ($p={p_endo}$)')

ax1.set_xlabel('x')
ax1.set_ylabel('y or t')

ax2.set_xlabel('x')
ax2.set_ylabel('y or t')

# colorbars
cbar1 = plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
cbar2 = plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)

ticks = np.arange(0, 1.01, 0.1)
for cbar in [cbar1, cbar2]:
    cbar.set_ticks(ticks)
    cbar.set_ticklabels([f'{t:.1f}' for t in ticks])

# ===== 3D plots =====
ax3 = fig.add_subplot(2, 2, 3, projection='3d')
surf1 = ax3.plot_surface(X, Y, df_exo_norm, cmap=custom_cmap, norm=norm,
                         linewidth=0, antialiased=True)
ax3.set_title('(c) Exogenous (3D view)')
ax3.view_init(elev=30, azim=135)

ax4 = fig.add_subplot(2, 2, 4, projection='3d')
surf2 = ax4.plot_surface(X, Y, df_endo_norm, cmap=custom_cmap, norm=norm,
                         linewidth=0, antialiased=True)
ax4.set_title('(d) Endogenous (3D view)')
ax4.view_init(elev=30, azim=135)

ax3.invert_yaxis()
ax4.invert_yaxis()

ax3.set_xlabel('x')
ax3.set_ylabel('y or t')
ax3.set_zlabel('A(x,t)')

ax4.set_xlabel('x')
ax4.set_ylabel('y or t')
ax4.set_zlabel('A(x, y or t)')

plt.tight_layout()
plt.savefig("./data/exo_endo_stm_model_1.png", dpi=300, bbox_inches='tight')  
plt.show()

In [ ]:
import random
import itertools

colors = ['tab:blue', 'tab:orange', 'tab:green']
linestyles = ['--', '-.']

# =========================
# seleção de linhas (equiespaçadas)
# =========================
ny = df_exo_norm.shape[0]
indices = [100, 170, 205,225]

slice_data = df_exo_norm[indices]
x = np.arange(df_exo_norm.shape[1])

# =========================
# figura com controle de largura
# =========================
fig, (ax1, ax2) = plt.subplots(
    1, 2,
    figsize=(12, 4),
    gridspec_kw={'width_ratios': [3, 7]}
)

# =========================
# 2D campo
# =========================
im = ax1.imshow(
    df_exo_norm,
    cmap=custom_cmap,
    norm=norm,
    origin='lower',
    aspect='auto'
)

ax1.set_title(f'(a) Exogenous ($p={p_exo}$)')
ax1.set_xlabel('x')
ax1.set_ylabel('y or t')


# destacar linhas selecionadas
for idx in indices:
    ax1.axhline(idx, color='red', alpha=0.3, linewidth=0.8)

plt.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)

# =========================
# séries deslocadas
# =========================
all_combinations = list(itertools.product(colors, linestyles))
random.shuffle(all_combinations)
print(all_combinations)

for i, (row, idx) in enumerate(zip(slice_data, indices)):
    color, linestyle = all_combinations[i % len(all_combinations)]

    ax2.plot(
        x,
        row,
        color=color,
        linestyle=linestyle,
        linewidth=1.8,
        label=f't = {idx}'
    )

ax2.set_title('(b) Temporal slices')
ax2.set_xlabel('x')
ax2.set_ylabel('A(x, y or t)')
ax2.grid(alpha=0.3)
ax2.set_axisbelow(False)

# legenda
ax2.legend(
    ncol=2,
    frameon=False
)

# layout
plt.subplots_adjust(wspace=0.3)
plt.tight_layout()
plt.savefig("./data/exo_stm_model_1_temporal_slices.png", dpi=300, bbox_inches='tight')  
plt.show()

In [ ]:

slice_data = df_endo_norm[indices]
x = np.arange(df_endo_norm.shape[1])

fig, (ax1, ax2) = plt.subplots(
    1, 2,
    figsize=(12, 4),
    gridspec_kw={'width_ratios': [3, 7]}
)

# 2D campo
im = ax1.imshow(
    df_endo_norm,
    cmap=custom_cmap,
    norm=norm,
    origin='lower',
    aspect='auto'
)

ax1.set_title(f'(a) Endogenous ($p={p_endo}$)')
ax1.set_xlabel('x')
ax1.set_ylabel('y or t')


# destacar linhas selecionadas
for idx in indices:
    ax1.axhline(idx, color='blue', alpha=0.3, linewidth=0.8)

plt.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)

# séries deslocadas
all_combinations = list(itertools.product(colors, linestyles))
random.shuffle(all_combinations)
print(all_combinations)

for i, (row, idx) in enumerate(zip(slice_data, indices)):
    color, linestyle = all_combinations[i % len(all_combinations)]

    ax2.plot(
        x,
        row,
        color=color,
        linestyle=linestyle,
        linewidth=1.8,
        label=f't = {idx}'
    )

ax2.set_title('(b) Temporal slices')
ax2.set_xlabel('x')
ax2.set_ylabel('A(x, y or t)')
ax2.grid(alpha=0.3)
ax2.set_axisbelow(False)

# legenda
ax2.legend(
    ncol=2,
    frameon=False
)

# layout
plt.subplots_adjust(wspace=0.3)
plt.tight_layout()
plt.savefig("./data/endo_stm_model_1_temporal_slices.png", dpi=300, bbox_inches='tight')  
plt.show()

## STM-Model (2+1)D

A generalização proposta estende o modelo para um campo espaço-temporal definido por:

[
A(t,x,y)
]

onde `x` e `y` representam as dimensões espaciais e `t` representa o tempo.

Cada instante `t` corresponde a um campo bidimensional multifractal, e a sequência desses campos descreve a evolução temporal do sistema. Dessa forma, o modelo representa uma cascata multifractal **2D no espaço + 1D no tempo**.

Essa abordagem permite simular estruturas mais próximas de sistemas físicos reais, como turbulência e campos geofísicos.

![Esquema STM 2+1D](https://drive.google.com/uc?export=view&id=1jN-euZhssp-Vz9-H3Y42PxObmOmvQ6ju)

---

## Implementação do STM-Model

O modelo foi construído preservando a lógica multiplicativa do **P-model**, adicionando evolução temporal e interação espacial entre os pontos da malha.

In [ ]:
def stm_model_3d(n, p=0.5, sigma=10):
    field = np.ones((1, 1, 1))

    for _ in range(n):
        field = next_step_3d(field, p, sigma)

    field = gaussian_filter(field, sigma=sigma)

    return field


def next_step_3d(field, p, sigma):
    nx, ny, nz = field.shape
    new_field = np.zeros((2*nx, 2*ny, 2*nz))

    for i in range(nx):
        for j in range(ny):
            for k in range(nz):

                base = field[i, j, k]

                px = p if np.random.rand() < 0.5 else (1-p)
                py = p if np.random.rand() < 0.5 else (1-p)
                pz = p if np.random.rand() < 0.5 else (1-p)

                weights = 2*np.array([
                    px*py*pz,
                    px*py*(1-pz),
                    px*(1-py)*pz,
                    px*(1-py)*(1-pz),
                    (1-px)*py*pz,
                    (1-px)*py*(1-pz),
                    (1-px)*(1-py)*pz,
                    (1-px)*(1-py)*(1-pz),
                ])

                idx = 0
                for di in range(2):
                    for dj in range(2):
                        for dk in range(2):
                            new_field[2*i+di, 2*j+dj, 2*k+dk] = base * weights[idx]
                            idx += 1

    return new_field

In [ ]:
p_exo = 0.18
np.random.seed(351)
df_exo = stm_model_3d(8, p_exo)

p_endo = 0.48
np.random.seed(351)
df_endo = stm_model_3d(8, p_endo)

# normalização entre 0 e 1
df_exo_norm = (df_exo - df_exo.min()) / (df_exo.max() - df_exo.min())
df_endo_norm = (df_endo - df_endo.min()) / (df_endo.max() - df_endo.min())

In [ ]:

# slice central de cada campo
# best_exo = df_exo_norm.shape[2] // 2
# best_endo = df_endo_norm.shape[2] // 2

# encontrar o slice z com maior valor máximo
max_por_slice_exo = df_exo_norm.max(axis=(0,1))
max_por_slice_endo = df_endo_norm.max(axis=(0,1))

best_exo = np.argmax(max_por_slice_exo)
best_endo = np.argmax(max_por_slice_endo)

slice_exo = df_exo_norm[:, :, best_exo]
slice_endo = df_endo_norm[:, :, best_endo]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im1 = axes[0].imshow(slice_exo, cmap=custom_cmap, origin='lower', vmin=0, vmax=1)
axes[0].set_title(f'Exo (p = {p_exo}), Frame = {best_exo}')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)

im2 = axes[1].imshow(slice_endo, cmap=custom_cmap, origin='lower', vmin=0, vmax=1)
axes[1].set_title(f'Endo (p = {p_endo}), Frame = {best_endo}')
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
slice_exo

In [ ]:
slice_endo

In [ ]:
from matplotlib import animation
from IPython.display import HTML

In [ ]:
plt.rcParams['animation.embed_limit'] = 200

# usar os dados normalizados diretamente
data_exo = df_exo_norm
data_endo = df_endo_norm

# dimensões
frame_height, frame_width = data_exo[:, :, 0].shape
x_coords = np.arange(frame_width)
y_coords = np.arange(frame_height)
X, Y = np.meshgrid(x_coords, y_coords)

# escala fixa de 0 a 1
zmin_exo, zmax_exo = 0, 1
zmin_endo, zmax_endo = 0, 1

# figura com 2 gráficos 3D
fig = plt.figure(figsize=(10, 6))
ax1 = fig.add_subplot(121, projection='3d')
ax2 = fig.add_subplot(122, projection='3d')

def init():
    ax1.clear()
    ax2.clear()

    surf1 = ax1.plot_surface(X, Y, data_exo[:, :, 0], cmap=custom_cmap, vmin=0, vmax=1)
    surf2 = ax2.plot_surface(X, Y, data_endo[:, :, 0], cmap=custom_cmap, vmin=0, vmax=1)

    ax1.set_zlim(0, 1)
    ax2.set_zlim(0, 1)

    ax1.set_title("Exo - Frame 0")
    ax2.set_title("Endo - Frame 0")

    ax1.set_xlabel('x')
    ax1.set_ylabel('y')
    ax2.set_xlabel('x')
    ax2.set_ylabel('y')

    ax1.invert_xaxis()
    ax2.invert_xaxis()

    return surf1, surf2

def update(frame_index):
    ax1.clear()
    ax2.clear()

    surf1 = ax1.plot_surface(X, Y, data_exo[:, :, frame_index], cmap=custom_cmap, vmin=0, vmax=1)
    surf2 = ax2.plot_surface(X, Y, data_endo[:, :, frame_index], cmap=custom_cmap, vmin=0, vmax=1)

    ax1.set_zlim(0, 1)
    ax2.set_zlim(0, 1)

    ax1.set_title(f"Exo - Frame {frame_index}")
    ax2.set_title(f"Endo - Frame {frame_index}")

    ax1.set_xlabel('x')
    ax1.set_ylabel('y')
    ax2.set_xlabel('x')
    ax2.set_ylabel('y')

    ax1.invert_xaxis()
    ax2.invert_xaxis()

    return surf1, surf2

ani = animation.FuncAnimation(
    fig, update,
    frames=data_exo.shape[2],
    init_func=init,
    interval=200,
    blit=False
)

ani.save("./data/exo_endo_stm_model_2.mp4", writer="ffmpeg", fps=20)

HTML(ani.to_jshtml())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# frames desejados
frames = [0, 75, 145, 210, 237, 255]

# grid espacial
frame_height, frame_width = data_exo[:, :, 0].shape
x_coords = np.arange(frame_width)
y_coords = np.arange(frame_height)
X, Y = np.meshgrid(x_coords, y_coords)

l = ["a", "b", "c", "d", "e", "f"]
# FUNÇÃO PARA PLOTAR
def plot_3d_grid(data, path):
    fig = plt.figure(figsize=(14, 8))

    for i, frame in enumerate(frames):
        ax = fig.add_subplot(2, 3, i+1, projection='3d')

        surf = ax.plot_surface(
            X, Y, data[:, :, frame],
            cmap=custom_cmap,
            vmin=0, vmax=1,
            linewidth=0,
            antialiased=True
        )

        ax.set_title(f'({l[i]}) t = {frame}')

        ax.set_xlabel('x')
        ax.set_ylabel('y')

        ax.set_zlim(0, 1)
        ax.invert_yaxis()
        ax.view_init(elev=30, azim=135)

    plt.tight_layout()

    plt.savefig(f"./data/{path}", dpi=300, bbox_inches='tight')  
    plt.show()

# EXO
plot_3d_grid(data_exo, 'exo_stm_model_2_frames.png')

# ENDO
plot_3d_grid(data_endo, 'endo_stm_model_2_frames.png')


## Considerações finais

O **STM-Model 2+1D** é uma extensão natural das cascatas multifractais clássicas para domínios espaço-temporais. O modelo mantém as propriedades multifractais do **P-model**, incorporando dependência espacial e evolução temporal, tornando-se adequado para a simulação de sistemas complexos.